## Pipeline

In [7]:
import sys
print(sys.version)

3.9.15 | packaged by conda-forge | (main, Nov 22 2022, 08:45:29) 
[GCC 10.4.0]


In [8]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path
import sys
from typing import List

from osgeo import gdal

# Set up the environment to point to the local repository.
nobackup = Path(os.environ['NOBACKUP'])
repoPath = nobackup / 'innovation-lab-repositories/lfm'

if not repoPath.exists():
    raise ValueError('Invalid path to repository.')
    
# Append the parent directory so 'lfm' is recognized as a module
sys.path.append(str(repoPath.parent))

from lfm.model.Pipeline import Pipeline

# ----------------------------------------------------------------------------
# getTileDbPath
# ----------------------------------------------------------------------------
def getTileDbPath(imageDir: Path, dbName: str = 'output_index.shp') -> Path:

    if not imageDir or not imageDir.exists() or not imageDir.is_dir():
        raise ValueError('A valid image directory must be provided.')

    fullPath: Path = imageDir / dbName

    if fullPath.exists():
        return fullPath

    MOON_SRS = 'GEOGCRS["Moon (2015) - Sphere / Ocentric", DATUM["Moon (2015) - Sphere", ELLIPSOID["Moon (2015) - Sphere",1737400,0, LENGTHUNIT["metre",1]]], PRIMEM["Reference Meridian",0, ANGLEUNIT["degree",0.0174532925199433]], CS[ellipsoidal,2], AXIS["geodetic latitude (Lat)",north, ORDER[1], ANGLEUNIT["degree",0.0174532925199433]], AXIS["geodetic longitude (Lon)",east, ORDER[2], ANGLEUNIT["degree",0.0174532925199433]], ID["IAU",30100,2015], REMARK["Source of IAU Coordinate systems: https://doi.org/10.1007/s10569-017-9805-5"]]'
    outFile: Path = imageDir / dbName
    
    # The database does not exist, so create it for the image directory.
    gdal.TileIndex(outFile, list(imageDir.glob('*.tif')), outputSRS=MOON_SRS)
    outFile.chmod(666)

    return outFile
      

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
imageDir = Path('/explore/nobackup/projects/ilab/projects/Lunar_FM/data/testdata/NAC_PHO/NAC_PHO_E186N1213/WAC/')
outDir = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/newCubes')
zone = '37S'
zoomLevel = 5

tileDbPath = getTileDbPath(imageDir)
pipeline = Pipeline(tileDbPath, outDir)

cubeFiles1: list[Path] = pipeline.runTileIndex(0, 0, zone, zoomLevel)
print('cubeFiles1:', cubeFiles1)

cubeFiles2: list[Path] = pipeline.runTileIndex(9, 0, zone, zoomLevel)
print('cubeFiles2:', cubeFiles2)

cubeFiles3: list[Path] = pipeline.runTileIndex(6, 10, zone, zoomLevel)
print('cubeFiles3:', cubeFiles3)

pipeline = Pipeline(tileDbPath, outDir)
cubeFiles4: list[Path] = pipeline.runTileIndex(9, 0, zone, 2)
print('cubeFiles4:', cubeFiles4)


## Include Other Image Types
- /explore/nobackup/projects/ilab/projects/Lunar_FM/data/testdata/Diviner/Hpar
- /explore/nobackup/projects/ilab/projects/Lunar_FM/data/testdata/gravity

In [ ]:
imageDir = Path('/explore/nobackup/projects/ilab/projects/Lunar_FM/data/testdata/Diviner/Hpar')
outDir = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/newCubes/data2')
zone = '37S'
zoomLevel = 5

rm = ResamplingMethod.AVERAGE
tileDbPath = getTileDbPath(imageDir)
pipeline = Pipeline(tileDbPath, outDir)
cubeFile5: Path = pipeline.runTileIndex(0, 0, zone, zoomLevel, rm)
print('cubeFile5:', cubeFile5)

imageDir = Path('/explore/nobackup/projects/ilab/projects/Lunar_FM/data/testdata/gravity')
tileDbPath = getTileDbPath(imageDir)
pipeline = Pipeline(tileDbPath, outDir)
cubeFile6: Path = pipeline.runTileIndex(9, 0, zone, zoomLevel, rm)
print('cubeFile6:', cubeFile6)


## Process New Data

### LRO_NAC_Pho_Sites
For the NAC and WAC data, a point 149.8 deg E/1.2 deg N was given.  This translates to tile(1, 63).

In [ ]:
imageDir = Path('/explore/nobackup/projects/lfm/processed_data/Lunar/LRO_NAC_Pho_Sites')
outDir = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/newCubes/nac')
zone = '42N'
zoomLevel = 5

lon = 149.8
lat = 1.2

rm = ResamplingMethod.AVERAGE
tileDbPath = getTileDbPath(imageDir)
pipeline = Pipeline(tileDbPath, outDir)
cubeFile7: Path = pipeline.runPoint(lat, lon, zone, zoomLevel, rm)
print('cubeFile7:', cubeFile7)


### LRO_WAC_Pho_Sites

In [ ]:
imageDir = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/LRO_WAC_Pho_Sites')
outDir = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/newCubes/wac')
zone = '42N'
zoomLevel = 5
rm = ResamplingMethod.AVERAGE

lon = 149.8
lat = 1.2

for subdir in sorted(imageDir.iterdir()):

    if subdir.is_dir():
 
        print(f'Processing subdirectory: {subdir.name}')

        dbPath = getTileDbPath(subdir)
 
        # Make an output directory for this group.
        groupOutDir = outDir / subdir.stem
        groupOutDir.mkdir(exist_ok=True)
        
        pipeline = Pipeline(dbPath, groupOutDir)
        cubeFile: Path = pipeline.runPoint(lat, lon, zone, zoomLevel, rm)


## Area of Interest
Instead of providing a tile index, an area of interest can be used.  This discovers all tiles intersecting the area and processes them.

Note: Currently, the only zoom level available to process by area of interest is level 1.  We are devising a way to handle the large amount of data required to make a global tile index database that includes all 26 zoom levels.

For the NAC and WAC data, a point 149.8 deg E/1.2 deg N was previously given.  We arbitrarily chose an AoI around this.
- Upper left: (149.7, 1.3)
- Lower right: (149.9, 1.1)

In [ ]:
imageDir = Path('/explore/nobackup/projects/lfm/processed_data/Lunar/LRO_NAC_Pho_Sites')
outDir = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/newCubes/nac')

ulLat = 1.3
ulLon = 149.7
lrLat = 1.1
lrLon = 149.9
zoomLevel = 1

tileDbPath = getTileDbPath(imageDir)
pipeline = Pipeline(tileDbPath, outDir)

cubeFiles: List[Path] = pipeline.run(ulLat, 
                                     ulLon, 
                                     lrLat, 
                                     lrLon, 
                                     zoomLevel)

print('cubeFiles:', cubeFiles)


In [9]:
imageDir = Path('/explore/nobackup/projects/lfm/processed_data/Lunar/LRO_WAC_Pho_Sites')
outDir = Path('/explore/nobackup/people/rlgill/SystemTesting/lunar/tileShift/wac')

# ulLat = 1.3
# ulLon = 149.7
# lrLat = 1.1
# lrLon = 149.9

# ulLat = 31.241501479374616
# ulLon = 121.01616204071722
# lrLat = 30.201163639632075
# lrLon = 122.22643208573085

# PID: M1365092036CE
# ulLat = 20.825151580726427
# ulLon = 122.12848622880361
# lrLat = 19.807219376083147
# lrLon = 123.21401751497638

ulLat = 31.241501479374616
ulLon = 121.01616204071722
lrLat = 30.201163639632075
lrLon = 122.22643208573085

zoomLevel = 5

tileDbPath = getTileDbPath(imageDir)
pipeline = Pipeline(tileDbPath, outDir, debug=False)

cubeFiles: List[Path] = pipeline.run(ulLat, 
                                     ulLon, 
                                     lrLat, 
                                     lrLon, 
                                     zoomLevel)

print('cubeFiles:', cubeFiles)


Found intersection: 38N
Processing (3, 39) / zone 38N / zoom 5
Total product IDs: 370
Total product IDs: 63
Processing (4, 39) / zone 38N / zoom 5
Total product IDs: 536
Total product IDs: 63
Processing (3, 40) / zone 38N / zoom 5
Total product IDs: 390
Total product IDs: 63
Processing (4, 40) / zone 38N / zoom 5
Total product IDs: 569
Total product IDs: 63
cubeFiles: [PosixPath('/explore/nobackup/people/rlgill/SystemTesting/lunar/tileShift/wac/Cube-LTM38N_Zoom-5_Tile-3-39_ProdId-M1095671530CE.tif'), PosixPath('/explore/nobackup/people/rlgill/SystemTesting/lunar/tileShift/wac/Cube-LTM38N_Zoom-5_Tile-3-39_ProdId-M1095678664CE.tif'), PosixPath('/explore/nobackup/people/rlgill/SystemTesting/lunar/tileShift/wac/Cube-LTM38N_Zoom-5_Tile-3-39_ProdId-M1095685801CE.tif'), PosixPath('/explore/nobackup/people/rlgill/SystemTesting/lunar/tileShift/wac/Cube-LTM38N_Zoom-5_Tile-3-39_ProdId-M1095692965CE.tif'), PosixPath('/explore/nobackup/people/rlgill/SystemTesting/lunar/tileShift/wac/Cube-LTM38N_Zoo